# DLAI Model Merging - Layer-adaptive improvement

The layer ablation suggested that interference is cumulative and that early blocks are the most damaging isolated group. This experiment tests a conservative remedy: attenuate only the merged update in BERT layers 0-1.

Seed 42 is development data. It selects one global factor for adaptive Mean and one for adaptive TIES from `{0.25, 0.50, 0.75, 1.00}` across all six task pairs. Factor 1.00 is the no-adaptation control. The choices are then frozen and evaluated on held-out seeds 7 and 123. Only held-out results support the improvement claim.

Before running, use **Add Input** to attach notebook 02's saved Output containing `pilot_specialists_seed42.zip`. Select **GPU T4 x2**, enable Internet, and Run All.

In [ ]:
!nvidia-smi
!find /kaggle/input -maxdepth 3 -type f | head -50

## Install the project and load development specialists

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
REPO = 'https://github.com/LeuxLello/Dlai-model-merging.git'
BRANCH = 'codex/multiseed-results'
WORKDIR = Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(WORKDIR)])
sys.path.insert(0,str(WORKDIR/'src')); os.chdir(WORKDIR)
COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(); print('Commit:',COMMIT)
bundles = list(Path('/kaggle/input').rglob('pilot_specialists_seed42.zip'))
assert bundles, 'Attach notebook-02 Output containing pilot_specialists_seed42.zip.'
DEV_ROOT = Path('/kaggle/working/pilot_specialists_seed42')
if DEV_ROOT.exists(): shutil.rmtree(DEV_ROOT)
with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(DEV_ROOT)
print('Seed-42 encoders:',len(list(DEV_ROOT.rglob('encoder.pt'))))

## Environment and shared helpers

In [ ]:
import gc, itertools, json, platform
import numpy as np, pandas as pd, torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.ablation import bert_mini_scopes, scale_merged_update_by_scope
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import mean_merge, ties_merge
from dlai_merge.training import TrainConfig, train_specialist
assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
GPU = torch.cuda.get_device_name(0); CAPABILITY = torch.cuda.get_device_capability(0)
assert CAPABILITY[0] >= 7, 'Use GPU T4 x2, not P100.'
torch.ones(1,device='cuda').add_(1); print('GPU:',GPU,'| capability:',CAPABILITY)
TASKS=['sst2','imdb','mrpc','rte']; PAIRS=list(itertools.combinations(TASKS,2))
BASE='prajjwal1/bert-mini'; DEV_SEED=42; CONFIRM_SEEDS=[7,123]
FACTORS=[0.25,0.50,0.75,1.00]
base_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2)
base_encoder={k:v.detach().cpu().clone() for k,v in base_model.base_model.state_dict().items()}
scopes=bert_mini_scopes(base_encoder.keys())
primitive={'embeddings':scopes['embeddings'],'early':scopes['early'],'late':scopes['late']}
flat=[key for keys in primitive.values() for key in keys]
assert len(flat)==len(set(flat))==len(base_encoder) and set(flat)==set(base_encoder)
print({name:len(keys) for name,keys in primitive.items()})
def adaptive(merged,factor):
    return scale_merged_update_by_scope(base_encoder,merged,{
        'embeddings':(primitive['embeddings'],1.0),
        'early':(primitive['early'],factor),
        'late':(primitive['late'],1.0)})
def evaluate_state(rows,evaluators,references,seed,pair,method,factor,state,phase):
    for task in pair:
        result=evaluators[task].evaluate(state); score=result['primary_score']
        rows.append({'phase':phase,'seed':seed,'pair':'+'.join(pair),'task':task,'method':method,
                     'early_factor':factor,'primary_metric':evaluators[task].primary_metric,
                     'score':score,'specialist_score':references[task],
                     'retained':score/references[task]})

## Development selection on seed 42
A single factor is selected per algorithm using every pair, never a pair-specific factor. Selection maximizes mean retention, with worst-case retention as a deterministic tie-breaker.

In [ ]:
def dev_artifact(task,name):
    found=list(DEV_ROOT.rglob(f'{task}/seed-{DEV_SEED}/{name}')); assert len(found)==1
    return found[0]
dev_encoders={t:torch.load(dev_artifact(t,'encoder.pt'),map_location='cpu',weights_only=True) for t in TASKS}
dev_heads={t:torch.load(dev_artifact(t,'head.pt'),map_location='cpu',weights_only=True) for t in TASKS}
dev_evaluators={t:TaskEvaluator(t,dev_heads[t],max_eval_samples=2000,seed=DEV_SEED,output_root='/kaggle/working/adaptive-dev-eval') for t in TASKS}
dev_references={t:dev_evaluators[t].evaluate(dev_encoders[t])['primary_score'] for t in TASKS}
dev_rows=[]
for pair in PAIRS:
    states=[dev_encoders[t] for t in pair]
    standards={'mean':mean_merge(base_encoder,states),'ties':ties_merge(base_encoder,states,density=0.2,scale=1.0)}
    for method,state in standards.items():
        evaluate_state(dev_rows,dev_evaluators,dev_references,DEV_SEED,pair,method,1.0,state,'development')
        for factor in FACTORS:
            evaluate_state(dev_rows,dev_evaluators,dev_references,DEV_SEED,pair,'adaptive_'+method,factor,adaptive(state,factor),'development')
dev_results=pd.DataFrame(dev_rows); assert len(dev_results)==120
dev_selection=(dev_results[dev_results.method.str.startswith('adaptive_')]
 .groupby(['method','early_factor'],as_index=False).agg(mean_retained=('retained','mean'),worst_retained=('retained','min')))
selected={}
for method in ['adaptive_mean','adaptive_ties']:
    ranked=dev_selection[dev_selection.method==method].sort_values(['mean_retained','worst_retained','early_factor'],ascending=[False,False,False])
    selected[method]=float(ranked.iloc[0].early_factor)
print('Frozen factors:',selected); dev_selection

## Held-out confirmation on seeds 7 and 123
We now freeze the two selected factors. Each specialist is retrained with the original 400-step budget. No choice below depends on held-out scores.

In [ ]:
TRAIN_ROOT=Path('/kaggle/working/adaptive_confirm_specialists'); confirm_rows=[]; specialist_rows=[]
for seed in CONFIRM_SEEDS:
    print('\n===== HELD-OUT SEED',seed,'=====')
    for task in TASKS:
        summary=train_specialist(TrainConfig(task=task,output_root=str(TRAIN_ROOT),seed=seed,max_train_samples=12000,max_eval_samples=2000,epochs=3,max_steps=400,eval_steps=100,train_batch_size=32,eval_batch_size=64,learning_rate=2e-5))
        metric=summary['primary_metric']; specialist_rows.append({'seed':seed,'task':task,'primary_metric':metric,'score':summary['eval_metrics']['eval_'+metric]})
    encoders={t:torch.load(TRAIN_ROOT/t/f'seed-{seed}'/'encoder.pt',map_location='cpu',weights_only=True) for t in TASKS}
    heads={t:torch.load(TRAIN_ROOT/t/f'seed-{seed}'/'head.pt',map_location='cpu',weights_only=True) for t in TASKS}
    evaluators={t:TaskEvaluator(t,heads[t],max_eval_samples=2000,seed=seed,output_root=f'/kaggle/working/adaptive-confirm-eval-{seed}') for t in TASKS}
    references={t:evaluators[t].evaluate(encoders[t])['primary_score'] for t in TASKS}
    for pair in PAIRS:
        states=[encoders[t] for t in pair]
        standard_mean=mean_merge(base_encoder,states); standard_ties=ties_merge(base_encoder,states,density=0.2,scale=1.0)
        candidates={'mean':(standard_mean,1.0),'ties':(standard_ties,1.0),
                    'adaptive_mean':(adaptive(standard_mean,selected['adaptive_mean']),selected['adaptive_mean']),
                    'adaptive_ties':(adaptive(standard_ties,selected['adaptive_ties']),selected['adaptive_ties'])}
        for method,(state,factor) in candidates.items(): evaluate_state(confirm_rows,evaluators,references,seed,pair,method,factor,state,'held_out')
    for task in TASKS:
        shutil.rmtree(TRAIN_ROOT/task/f'seed-{seed}'/'trainer',ignore_errors=True)
    del encoders,heads,evaluators; gc.collect(); torch.cuda.empty_cache()
confirmation_results=pd.DataFrame(confirm_rows); assert len(confirmation_results)==96
print('Held-out task-level rows:',len(confirmation_results))

## Paired held-out analysis

In [ ]:
per_seed_pair=(confirmation_results.groupby(['seed','pair','method','early_factor'],as_index=False).agg(mean_retained=('retained','mean'),worst_retained=('retained','min')))
comparisons=[('adaptive_mean','mean'),('adaptive_ties','ties')]; paired_rows=[]; rng=np.random.default_rng(2026)
for adaptive_method,baseline_method in comparisons:
    a=per_seed_pair[per_seed_pair.method==adaptive_method][['seed','pair','mean_retained']].rename(columns={'mean_retained':'adaptive'})
    b=per_seed_pair[per_seed_pair.method==baseline_method][['seed','pair','mean_retained']].rename(columns={'mean_retained':'baseline'})
    paired=a.merge(b,on=['seed','pair']); delta=(paired.adaptive-paired.baseline).to_numpy()
    boot=delta[rng.integers(0,len(delta),size=(10000,len(delta)))].mean(axis=1)
    paired_rows.append({'adaptive_method':adaptive_method,'baseline_method':baseline_method,'selected_factor':selected[adaptive_method],
      'n_pair_seed_units':len(delta),'mean_delta':delta.mean(),'median_delta':np.median(delta),'win_rate':(delta>0).mean(),
      'tie_rate':np.isclose(delta,0,atol=1e-12).mean(),'worst_delta':delta.min(),'best_delta':delta.max(),
      'bootstrap_ci_low':np.quantile(boot,0.025),'bootstrap_ci_high':np.quantile(boot,0.975)})
improvement_summary=pd.DataFrame(paired_rows)
method_summary=(per_seed_pair.groupby('method',as_index=False).agg(mean_retained=('mean_retained','mean'),between_unit_sd=('mean_retained','std'),worst_retained=('worst_retained','min')))
display(method_summary); display(improvement_summary)

## Figures and reproducible export

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
FIGURES=Path('/kaggle/working/adaptive_figures'); FIGURES.mkdir(exist_ok=True)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
sns.lineplot(data=dev_selection,x='early_factor',y='mean_retained',hue='method',marker='o',ax=axes[0]); axes[0].set_title('Development selection (seed 42)'); axes[0].axvline(1.0,color='grey',ls='--',lw=1)
plot_pairs=[]
for adaptive_method,baseline_method in comparisons:
    a=per_seed_pair[per_seed_pair.method==adaptive_method][['seed','pair','mean_retained']].rename(columns={'mean_retained':'adaptive'})
    b=per_seed_pair[per_seed_pair.method==baseline_method][['seed','pair','mean_retained']].rename(columns={'mean_retained':'baseline'})
    d=a.merge(b,on=['seed','pair']); d['comparison']=adaptive_method.replace('adaptive_',''); d['delta']=d.adaptive-d.baseline; plot_pairs.append(d)
delta_plot=pd.concat(plot_pairs); sns.stripplot(data=delta_plot,x='comparison',y='delta',hue='seed',dodge=True,ax=axes[1]); axes[1].axhline(0,color='black',lw=1); axes[1].set_title('Held-out paired changes')
fig.tight_layout(); figure=FIGURES/'layer_adaptive_improvement.png'; fig.savefig(figure,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
OUT=Path('/kaggle/working/layer_adaptive_improvement_results'); OUT.mkdir(exist_ok=True)
dev_results.to_csv(OUT/'development_results.csv',index=False); dev_selection.to_csv(OUT/'development_selection.csv',index=False)
confirmation_results.to_csv(OUT/'heldout_results.csv',index=False); per_seed_pair.to_csv(OUT/'heldout_per_seed_pair.csv',index=False)
pd.DataFrame(specialist_rows).to_csv(OUT/'heldout_specialists.csv',index=False); method_summary.to_csv(OUT/'heldout_method_summary.csv',index=False)
improvement_summary.to_csv(OUT/'improvement_summary.csv',index=False); (OUT/'selected_factors.json').write_text(json.dumps(selected,indent=2))
metadata={'purpose':'development-selected layer-adaptive merging with held-out confirmation','commit':COMMIT,'base_model':BASE,'tasks':TASKS,'development_seed':DEV_SEED,'heldout_seeds':CONFIRM_SEEDS,'factor_candidates':FACTORS,'selected_factors':selected,'gpu':GPU,'python':platform.python_version(),'torch':torch.__version__,'primary_claim_uses_heldout_only':True,'bootstrap_resamples':10000}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2)); shutil.copy2(figure,OUT/figure.name)
archive=shutil.make_archive('/kaggle/working/layer_adaptive_improvement_results','zip',OUT)
print(archive); print('Download this ZIP from the notebook Output tab.')